# Image → 3D Voxel Model — Phase 3 Starter

This notebook trains a small neural network that takes **one photo of an object**
and predicts its **3D shape** as a 32×32×32 voxel grid (a blocky 3D occupancy grid —
think Minecraft-block resolution). This is the classic beginner image-to-3D setup
(based on the 3D-R2N2 paper/dataset), and it's sized to run on a **free Colab GPU**.

**Runtime > Change runtime type > GPU (T4)** before running anything below.

Pipeline: `photo -> CNN encoder -> 3D decoder -> voxel grid -> mesh (.obj/.glb)`

If any cell errors, copy the FULL error message back to Claude and paste your code — don't guess.

In [ ]:
# Cell 1: Check GPU is active
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (go enable GPU in Runtime settings!)")

In [ ]:
# Cell 2: Install extra libraries we need
!pip install trimesh scikit-image -q
print("Done.")

## Step 1: Download the dataset

This pulls the official Stanford 3D-R2N2 dataset: rendered photos of ShapeNet objects
+ their matching 3D voxel shapes. This is a few GB total — it will take several minutes.
**Only run this once per Colab session** (it's lost when the session ends, so re-run only
if your runtime restarts).

In [ ]:
# Cell 3: Download dataset (takes ~10-20 min depending on Colab's connection)
import os

os.makedirs("/content/data", exist_ok=True)
%cd /content/data

if not os.path.exists("ShapeNetRendering"):
    !wget -q --show-progress http://cvgl.stanford.edu/data2/ShapeNetRendering.tgz
    !tar -xzf ShapeNetRendering.tgz
    !rm ShapeNetRendering.tgz
    print("Renderings ready.")
else:
    print("Renderings already downloaded.")

if not os.path.exists("ShapeNetVox32"):
    !wget -q --show-progress http://cvgl.stanford.edu/data2/ShapeNetVox32.tgz
    !tar -xzf ShapeNetVox32.tgz
    !rm ShapeNetVox32.tgz
    print("Voxels ready.")
else:
    print("Voxels already downloaded.")

## File structure after download

```
/content/data/
├── ShapeNetRendering/
│   └── <category_id>/           # e.g. 02691156 = airplane
│       └── <model_id>/
│           └── rendering/
│               ├── 00.png ... 23.png   # 24 photos of the same object
│               └── rendering_metadata.txt
└── ShapeNetVox32/
    └── <category_id>/
        └── <model_id>/
            └── model.binvox      # the matching 3D voxel shape
```

Each `model_id` folder in both places matches up — that's your (image, 3D shape) training pair.

To keep training fast on a free GPU, we'll start with **just 1-2 categories** instead of
all 13 (you can add more later once training works).

In [ ]:
# Cell 4: A parser for the .binvox voxel file format (no external library needed)
import numpy as np

def read_binvox(path):
    """Reads a .binvox file and returns a (32,32,32) numpy boolean array."""
    with open(path, "rb") as f:
        line = f.readline().strip()
        if not line.startswith(b"#binvox"):
            raise IOError("Not a binvox file")
        dims = list(map(int, f.readline().strip().split(b" ")[1:]))
        _ = f.readline()  # translate line
        _ = f.readline()  # scale line
        _ = f.readline()  # data line
        raw_data = np.frombuffer(f.read(), dtype=np.uint8)

    values, counts = raw_data[::2], raw_data[1::2]
    voxels = np.repeat(values, counts).astype(bool)
    voxels = voxels.reshape(dims)
    return voxels

print("binvox reader ready.")

In [ ]:
# Cell 5: Build the list of (image, voxel) training pairs for chosen categories
import glob

# ShapeNet category IDs. Start with just airplane + car (fast to train, visually clear).
# Full list of 13 category IDs is in the dataset's README if you want to add more later.
CATEGORIES = {
    "02691156": "airplane",
    "02958343": "car",
}

RENDER_ROOT = "/content/data/ShapeNetRendering"
VOXEL_ROOT = "/content/data/ShapeNetVox32"

samples = []
for cat_id in CATEGORIES:
    model_dirs = glob.glob(f"{RENDER_ROOT}/{cat_id}/*")
    for model_dir in model_dirs:
        model_id = os.path.basename(model_dir)
        voxel_path = f"{VOXEL_ROOT}/{cat_id}/{model_id}/model.binvox"
        if os.path.exists(voxel_path):
            samples.append({
                "render_dir": f"{model_dir}/rendering",
                "voxel_path": voxel_path,
                "category": CATEGORIES[cat_id],
            })

print(f"Found {len(samples)} training pairs across {len(CATEGORIES)} categories.")

In [ ]:
# Cell 6: Sanity check — visualize one sample (photo + its 3D voxel shape)
import matplotlib.pyplot as plt
from PIL import Image

sample = samples[0]
img = Image.open(f"{sample['render_dir']}/00.png").convert("RGB")
voxels = read_binvox(sample["voxel_path"])

fig = plt.figure(figsize=(10, 5))

ax1 = fig.add_subplot(1, 2, 1)
ax1.imshow(img)
ax1.set_title(f"Input photo ({sample['category']})")
ax1.axis("off")

ax2 = fig.add_subplot(1, 2, 2, projection="3d")
ax2.voxels(voxels, edgecolor="k", linewidth=0.1)
ax2.set_title("Ground-truth 3D shape")

plt.show()
print(f"Voxel grid shape: {voxels.shape}, occupied voxels: {voxels.sum()}")

## Step 2: Dataset + DataLoader

Wraps everything above into a proper PyTorch `Dataset` so we can batch and shuffle
during training.

In [ ]:
# Cell 7: PyTorch Dataset
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import random

class ImageVoxelDataset(Dataset):
    def __init__(self, samples, image_size=128):
        self.samples = samples
        self.transform = T.Compose([
            T.Resize((image_size, image_size)),
            T.ToTensor(),  # -> (3, H, W), values in [0,1]
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        view_idx = random.randint(0, 23)  # pick a random one of the 24 photos
        img = Image.open(f"{s['render_dir']}/{view_idx:02d}.png").convert("RGB")
        img_tensor = self.transform(img)

        voxels = read_binvox(s["voxel_path"]).astype("float32")
        voxel_tensor = torch.from_numpy(voxels).unsqueeze(0)  # (1, 32, 32, 32)

        return img_tensor, voxel_tensor

# 90/10 train/val split
random.seed(42)
random.shuffle(samples)
split = int(0.9 * len(samples))
train_samples, val_samples = samples[:split], samples[split:]

train_ds = ImageVoxelDataset(train_samples)
val_ds = ImageVoxelDataset(val_samples)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2)

print(f"Train samples: {len(train_ds)}, Val samples: {len(val_ds)}")

## Step 3: The model

A **2D CNN encoder** turns the photo into a compact feature vector. A **3D transposed-CNN
decoder** expands that vector into a 32×32×32 occupancy grid (probability that each voxel
is "filled"). This is the simplest architecture that actually works for this task —
it's the same idea 3D-R2N2 and its successors use, just without the recurrent multi-view part.

In [ ]:
# Cell 8: Model definition
import torch.nn as nn

class ImageEncoder(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.BatchNorm2d(32), nn.ReLU(),   # 128 -> 64
            nn.Conv2d(32, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),  # 64 -> 32
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),# 32 -> 16
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(),# 16 -> 8
            nn.AdaptiveAvgPool2d(1),  # -> (256, 1, 1)
        )
        self.fc = nn.Linear(256, latent_dim)

    def forward(self, x):
        x = self.conv(x).flatten(1)
        return self.fc(x)


class VoxelDecoder(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 256 * 4 * 4 * 4)
        self.deconv = nn.Sequential(
            nn.ConvTranspose3d(256, 128, 4, 2, 1), nn.BatchNorm3d(128), nn.ReLU(),  # 4 -> 8
            nn.ConvTranspose3d(128, 64, 4, 2, 1), nn.BatchNorm3d(64), nn.ReLU(),    # 8 -> 16
            nn.ConvTranspose3d(64, 1, 4, 2, 1),                                     # 16 -> 32
        )

    def forward(self, z):
        x = self.fc(z).view(-1, 256, 4, 4, 4)
        return self.deconv(x)  # raw logits, shape (B, 1, 32, 32, 32)


class ImageTo3D(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.encoder = ImageEncoder(latent_dim)
        self.decoder = VoxelDecoder(latent_dim)

    def forward(self, image):
        z = self.encoder(image)
        return self.decoder(z)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = ImageTo3D().to(device)
print(model)
print(f"\nTotal params: {sum(p.numel() for p in model.parameters()):,}")

## Step 4: Training loop

We treat this as a per-voxel binary classification problem (is this voxel filled or not?),
so we use `BCEWithLogitsLoss`. Expect **20-40 epochs** to get decent blocky shapes on this
small model/dataset — each epoch should take under a minute on a Colab T4 with 2 categories.

In [ ]:
# Cell 9: Training loop
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

EPOCHS = 30
history = {"train_loss": [], "val_loss": []}

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for imgs, voxels in train_loader:
        imgs, voxels = imgs.to(device), voxels.to(device)

        optimizer.zero_grad()
        pred = model(imgs)
        loss = criterion(pred, voxels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)

    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, voxels in val_loader:
            imgs, voxels = imgs.to(device), voxels.to(device)
            pred = model(imgs)
            val_loss += criterion(pred, voxels).item() * imgs.size(0)
    val_loss /= len(val_ds)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    print(f"Epoch {epoch+1}/{EPOCHS} | train loss: {train_loss:.4f} | val loss: {val_loss:.4f}")

# Save checkpoint so you don't lose progress if the session disconnects
torch.save(model.state_dict(), "/content/image_to_3d_model.pth")
print("\nSaved checkpoint to /content/image_to_3d_model.pth")

In [ ]:
# Cell 10: Plot training curves
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"], label="val")
plt.xlabel("epoch")
plt.ylabel("BCE loss")
plt.legend()
plt.title("Training progress")
plt.show()

## Step 5: Try it on a real prediction

Run the trained model on a validation photo and compare predicted vs. ground-truth shape.

In [ ]:
# Cell 11: Visualize a prediction vs ground truth
model.eval()
sample = val_samples[0]

img = Image.open(f"{sample['render_dir']}/00.png").convert("RGB")
img_tensor = train_ds.transform(img).unsqueeze(0).to(device)

with torch.no_grad():
    pred_logits = model(img_tensor)
    pred_probs = torch.sigmoid(pred_logits)[0, 0].cpu().numpy()
    pred_voxels = pred_probs > 0.5  # threshold to get a solid/empty grid

gt_voxels = read_binvox(sample["voxel_path"])

fig = plt.figure(figsize=(15, 5))
ax1 = fig.add_subplot(1, 3, 1)
ax1.imshow(img)
ax1.set_title("Input photo")
ax1.axis("off")

ax2 = fig.add_subplot(1, 3, 2, projection="3d")
ax2.voxels(gt_voxels, edgecolor="k", linewidth=0.1)
ax2.set_title("Ground truth")

ax3 = fig.add_subplot(1, 3, 3, projection="3d")
ax3.voxels(pred_voxels, edgecolor="k", linewidth=0.1, facecolors="orange")
ax3.set_title("Model prediction")

plt.show()

## Step 6: Export the prediction as a downloadable 3D file

Converts the blocky voxel grid into an actual mesh (using marching cubes) and saves it
as `.obj` and `.glb` — the same file types your website will eventually let users download.

In [ ]:
# Cell 12: Export prediction to .obj and .glb
from skimage import measure
import trimesh

def voxels_to_mesh(voxel_grid, threshold=0.5):
    # Pad by 1 voxel on each side so marching_cubes handles edge faces correctly
    padded = np.pad(voxel_grid.astype(float), 1, mode="constant")
    verts, faces, normals, _ = measure.marching_cubes(padded, level=threshold)
    mesh = trimesh.Trimesh(vertices=verts, faces=faces, vertex_normals=normals)
    return mesh

mesh = voxels_to_mesh(pred_probs)
mesh.export("/content/prediction.obj")
mesh.export("/content/prediction.glb")

print("Exported /content/prediction.obj and /content/prediction.glb")
print("Download them from the Colab file browser on the left (folder icon).")

---
## What to do next

1. Run every cell top to bottom. If something errors, copy the **full error text** and
   paste it back along with which cell — that's enough for Claude to fix it.
2. Once this trains successfully and predictions look roughly right (blocky but
   recognizable as a plane/car), try adding more categories in Cell 5, or bump
   `EPOCHS` up.
3. After that works end-to-end, that's your signal to move to Phase 4: fine-tuning a
   real pretrained model (TripoSR) for much higher quality output — the concepts here
   (encoder → 3D representation → decoder → mesh export) carry over directly.